# Loan Collections Prioritiser


Collections officers in microfinance typically sort delinquent borrowers by descending debt balance (*Biggest Debt First*).
This notebook walks through the machine learning pipeline that replaces this heuristic with an **Expected Recoverable Value** engine ($P(\text{cure}) \times \hat{r} \times \text{Debt}$), achieving **88.0% precision** (vs 7.0% incumbent) and **+2,164,156 FCFA (+65.6%)** in net cash recovery lift per 100 worked accounts.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add project root to sys.path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.data_loading import load_installments
from src.target import build_arrears_events, base_recovery_rate
from src.features import build_features, FEATURE_COLUMNS, FORBIDDEN_POST_CONTACT
from src.model import load_model, AdaptablePrioritiser, compute_xai_waterfall
from src.evaluate import evaluate_collections_policy
from src.contact import assign_bamenda_quarter, generate_bilingual_scripts, calculate_benskin_route_savings

print("✓ All Skye8 modules and dependencies successfully imported.")

✓ All Skye8 modules and dependencies successfully imported.


## 1. Load Processed Production Call Book (400 Accounts)
Let us inspect the ranked production book generated for the 3 collections officers in Bamenda.

In [2]:
data_file = project_root / "data" / "processed" / "weekly_400_accounts.csv"
df = pd.read_csv(data_file)
print(f"Loaded {len(df)} delinquent accounts across {len(df.columns)} multi-task features.")
cols_to_show = ["rank", "SK_ID_CURR", "assigned_officer", "bamenda_quarter", "overdue_amount", "recovery_probability", "partial_recovery_ratio", "recommended_channel", "expected_gross_recovery_fcfa"]
df[cols_to_show].head(10)

Loaded 400 delinquent accounts across 55 multi-task features.


,rank,SK_ID_CURR,assigned_officer,bamenda_quarter,overdue_amount,recovery_probability,partial_recovery_ratio,recommended_channel,expected_gross_recovery_fcfa
0,1,103380,Officer 1 - John N.,Commercial Avenue / Food Market (Zone 1),30613.410,0.997010,0.757945,CALL,23133.905828
1,2,102286,Officer 2 - Beatrice M.,Nkwen / Mile 2 - Mile 4 (Zone 2),12966.255,0.997442,0.764151,SMS,9882.834301
2,3,101516,Officer 3 - Emmanuel K.,Nkwen / Mile 2 - Mile 4 (Zone 2),12336.930,0.995592,0.765224,SMS,9398.905907
3,4,103176,Officer 1 - John N.,Nkwen / Mile 2 - Mile 4 (Zone 2),23376.060,0.582140,0.617736,CALL,8406.232591
4,5,100906,Officer 2 - Beatrice M.,Nkwen / Mile 2 - Mile 4 (Zone 2),9000.000,0.997455,0.762596,SMS,6845.893506
5,6,101134,Officer 3 - Emmanuel K.,Old Council / Hospital Roundabout (Zone 5),9000.000,0.997427,0.747202,SMS,6707.518960
6,7,104366,Officer 1 - John N.,Nkwen / Mile 2 - Mile 4 (Zone 2),9000.000,0.980011,0.752193,SMS,6634.413200
7,8,104226,Officer 2 - Beatrice M.,Nkwen / Mile 2 - Mile 4 (Zone 2),9000.000,0.986315,0.733758,SMS,6513.448555
8,9,103073,Officer 3 - Emmanuel K.,Mankon / Small Mankon (Zone 4),8100.000,0.995957,0.750399,SMS,6053.655137
9,10,103976,Officer 1 - John N.,Nkwen / Mile 2 - Mile 4 (Zone 2),7875.000,0.997449,0.769384,SMS,6043.444578


## 2. Quantitative Evaluation: Why Biggest Debt First Fails
Comparing the Skye8 Expected Value Prioritiser against the Incumbent heuristic on held-out test data.

In [3]:
import json
metrics_file = project_root / "reports" / "metrics_summary.json"
with open(metrics_file) as f:
    metrics = json.load(f)

k100 = metrics["k_evaluations"]["100"]
print(f"ROC-AUC: {metrics['roc_auc']:.4f} | PR-AUC: {metrics['pr_auc']:.4f}")
print(f"Precision @ 100 Accounts: {k100['precision_ev']:.1%} vs {k100['precision_incumbent']:.1%} (Incumbent)")
print(f"Relative Lift in Cures: {k100['precision_gain_ratio']:.2f}x")
print(f"Net Cash Lift: +{k100['money_lift_fcfa']:,.0f} FCFA (+{k100['money_lift_pct']:.1f}%)")

ROC-AUC: 0.9864 | PR-AUC: 0.9288
Precision @ 100 Accounts: 88.0% vs 7.0% (Incumbent)
Relative Lift in Cures: 12.57x
Net Cash Lift: +2,176,562 FCFA (+65.9%)


## 3. Explainable AI (XAI) Waterfall for Single Borrower
Inspecting the local feature contributions driving Client #103380 score.

In [4]:
sample = df.iloc[0]
xai = compute_xai_waterfall(sample)
print(f"Client #{sample['SK_ID_CURR']} - Score: {sample['recovery_probability']:.1%}")
print("\nTop Positive Factors:")
for p in xai["top_positive"]:
    print(f"  [+] {p['factor']} ({p['impact']}) -> {p['desc']}")
print("\nTop Negative Drag Factors:")
for n in xai["top_negative"]:
    print(f"  [-] {n['factor']} ({n['impact']}) -> {n['desc']}")

Client #103380 - Score: 99.7%

Top Positive Factors:
  [+] High Historical Repayment Compliance (+24.5%) -> Client paid 100% of past scheduled dues in full.
  [+] Modest Arrears Principal (+12.0%) -> Manageable overdue sum (30,613 FCFA) easily settled in single installment.

Top Negative Drag Factors:
  [-] Heavy Cross-Institutional Bureau Debt (-18.2%) -> Borrower carries 264,042 FCFA across 2 external loans.


## 4. Culturally Authentic Bilingual Outreach Scripts
Inspecting the generated scripts in Formal English and Cameroon Pidgin English.

In [5]:
eng_script, pid_script = generate_bilingual_scripts(
    account_id=int(sample["SK_ID_CURR"]),
    loan_id=int(sample["SK_ID_PREV"]),
    overdue_amount=float(sample["overdue_amount"]),
    assigned_officer=str(sample["assigned_officer"]),
    is_vulnerable=bool(sample["is_vulnerable"]),
    segment=str(sample["behavioral_segment"])
)
print("🇬🇧 FORMAL ENGLISH SCRIPT:")
print(eng_script)
print("\n🇨🇲 CAMEROON PIDGIN ENGLISH SCRIPT:")
print(pid_script)

🇬🇧 FORMAL ENGLISH SCRIPT:
Respected Member (Client #103380), this is John from your Credit Union Member Welfare desk. We acknowledge your valued loyalty. We noticed an overdue instalment of 30,613 FCFA on Loan Ref #2666240. Please be reassured that our cooperative offers flexible hardship restructuring. You are warmly invited to visit our branch or call us to arrange an amicable repayment schedule that protects your household.

🇨🇲 CAMEROON PIDGIN ENGLISH SCRIPT:
Mami/Papa (Client #103380), we salute you with plenty respect! Na John from Credit Union Welfare team. We know say sickness, market slowdown, or family matter fit hold person. Your loan ref #2666240 get small balance of 30,613 FCFA, but no worry or fret at all. Credit Union na our own family bank; when body strong, pass for branch make we sit down arrange small-small payment wey hand fit reach.


## 5. Motorcycle Route Clustering & Benskin Fuel Economy
Evaluating the transit fuel savings for Officer 3 (Emmanuel K.) across Bamenda quarters.

In [6]:
emmanuel_df = df[df["assigned_officer"].str.contains("Emmanuel", na=False)]
savings = calculate_benskin_route_savings(emmanuel_df)
print(f"Scheduled Field Visits: {savings['n_field_visits']}")
print(f"Unclustered Distance: {savings['unclustered_km']} km -> Clustered: {savings['clustered_km']} km")
print(f"Distance Cut: {savings['km_saved']} km (-{savings['pct_distance_saved']}%) ")
print(f"Fuel Saved: {savings['fuel_saved_liters']} Litres (@ 850 FCFA/L = {savings['fuel_cash_saved_fcfa']:,.0f} FCFA saved)")

Scheduled Field Visits: 0
Unclustered Distance: 0.0 km -> Clustered: 0.0 km
Distance Cut: 0.0 km (-0.0%) 
Fuel Saved: 0.0 Litres (@ 850 FCFA/L = 0 FCFA saved)


## 6. Adaptable Prioritiser Demonstration
Demonstrating online Bayesian adaptation as collections officers record outcomes.

In [7]:
engine = AdaptablePrioritiser()
print(f"Baseline Prior Odds Offset: {engine.branch_offset}")
engine.adapt_to_branch("Urban Commercial (Mankon / Commercial Ave)")
print(f"Adapted Urban Commercial Offset: {engine.branch_offset:+.2f}")
delta = engine.incorporate_field_feedback(int(sample["SK_ID_CURR"]), "Cured & Settled in Full")
print(f"Incorporated field feedback for Client #{sample['SK_ID_CURR']}: delta = {delta:+.1%}")

Baseline Prior Odds Offset: 0.0
Adapted Urban Commercial Offset: +0.18
Incorporated field feedback for Client #103380: delta = +40.0%
